# FIFA World Cup 2026 - Objective 1 Analysis
Four analytic tasks using 2026 World Cup data:
1. Do host teams score more goals than non-host teams?
2. Do UEFA teams get fewer yellow cards than CONMEBOL teams?
3. Do goalkeepers on knockout-stage teams have a higher save % than teams out in the group stage?
4. Do substitutes score/assist less per 90 minutes than starters?

## Submitted By : Marium Akter Mim s397784

### I Used FBref. Manually converted to csv file . As there is no way to download directly csv file format. 

In [3]:

import pandas as pd
import numpy as np
from scipy import stats


In [4]:

np.random.seed(7)
folder = "."


## Step 1: Load the data files
A couple of the FBref files have two header rows, so the first row is skipped.

In [5]:

squad_stats = pd.read_csv(folder + "/squad_standard_stats.csv", skiprows=1)
cards = pd.read_csv(folder + "/misc_cards.csv", skiprows=1)


In [6]:

gk_stats = pd.read_csv(folder + "/keepers.csv", skiprows=1)
matches = pd.read_csv(folder + "/schedule.csv")


In [7]:

player_time = pd.read_csv(folder + "/player_playing_time.csv", skiprows=1)
player_stats = pd.read_csv(folder + "/player_standard_stats.csv", skiprows=1)


In [8]:

# the schedule file has some empty rows used as spacers, remove them
matches = matches.dropna(subset=["Home", "Away", "Score"])
matches.head()


,Round,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,Referee,Match Report,Notes
0,Group stage,1.0,Thu,2026-06-11,13:00 (00:00),Mexico mx,2–0,za South Africa,80824.0,Estadio Banorte (Neutral Site),Wilton Sampaio,Match Report,NaN
1,Group stage,1.0,Thu,2026-06-11,20:00 (07:00),Korea Republic kr,2–1,cz Czechia,44985.0,Estadio Akron (Neutral Site),Amin Omar,Match Report,NaN
2,Group stage,1.0,Fri,2026-06-12,15:00 (00:00),Canada ca,1–1,ba Bosnia–Herz,43002.0,BMO Field (Neutral Site),Facundo Tello,Match Report,NaN
3,Group stage,1.0,Fri,2026-06-12,18:00 (06:00),USA us,4–1,py Paraguay,70492.0,SoFi Stadium (Neutral Site),Danny Makkelie,Match Report,NaN
4,Group stage,1.0,Sat,2026-06-13,12:00 (00:00),Qatar qa,1–1,ch Switzerland,67966.0,Levi's Stadium (Neutral Site),Said Martínez,Match Report,NaN


## Step 2: Clean up the team names
In the raw files, team names have a small country code attached, like `dz Algeria` or `Mexico mx`. The function below removes that code so all files use the same team name.

In [9]:

def clean_name(name):
    words = str(name).split(" ")
    if len(words[0]) <= 3 and words[0].islower():
        return " ".join(words[1:])
    if len(words[-1]) <= 3 and words[-1].islower():
        return " ".join(words[:-1])
    return name


In [10]:

squad_stats["Squad"] = squad_stats["Squad"].apply(clean_name)
cards["Squad"] = cards["Squad"].apply(clean_name)


In [11]:

gk_stats["Squad"] = gk_stats["Squad"].apply(clean_name)
player_time["Squad"] = player_time["Squad"].apply(clean_name)
player_stats["Squad"] = player_stats["Squad"].apply(clean_name)


In [12]:

matches["home_team"] = matches["Home"].apply(clean_name)
matches["away_team"] = matches["Away"].apply(clean_name)


Next, the score column (like `2-0`) is split into two separate number columns.

In [13]:

def get_scores(score_text):
    score_text = score_text.replace("\u2013", "-")
    for part in score_text.split(" "):
        if "-" in part and "(" not in part:
            home_goals, away_goals = part.split("-")
            return int(home_goals), int(away_goals)


In [14]:

matches["home_goals"] = matches["Score"].apply(lambda x: get_scores(x)[0])
matches["away_goals"] = matches["Score"].apply(lambda x: get_scores(x)[1])
matches[["home_team", "home_goals", "away_team", "away_goals"]].head()


,home_team,home_goals,away_team,away_goals
0,Mexico,2,South Africa,0
1,Korea Republic,2,Czechia,1
2,Canada,1,Bosnia–Herz,1
3,USA,4,Paraguay,1
4,Qatar,1,Switzerland,1


## Two small functions used again in every task
One prints basic descriptive stats, the other calculates a 95% confidence interval.

In [15]:

def print_stats(sample, label):
    print(label)
    print("count:", len(sample))
    print("mean:", round(sample.mean(), 2))
    print("median:", round(sample.median(), 2))
    print("std dev:", round(sample.std(), 2))
    print()


In [16]:

def get_ci(sample):
    n = len(sample)
    mean = sample.mean()
    std_error = stats.sem(sample)
    margin = std_error * stats.t.ppf(0.975, n - 1)
    return mean - margin, mean + margin


---
## Task 1: Host teams vs non-host teams (goals per match)

Population = every team's goal count from every match (208 rows, since 104 matches x 2 teams each). Only 3 teams are hosts (Canada, Mexico, USA), so all of their match records are used. For the non-host teams, a random sample of 40 is taken.

In [17]:

host_countries = ["Canada", "Mexico", "USA"]


In [18]:

home_part = matches[["home_team", "home_goals"]]
home_part.columns = ["team", "goals"]


In [19]:

away_part = matches[["away_team", "away_goals"]]
away_part.columns = ["team", "goals"]


In [20]:

team_goals = pd.concat([home_part, away_part])
team_goals["is_host"] = team_goals["team"].isin(host_countries)
team_goals.head()


,team,goals,is_host
0,Mexico,2,True
1,Korea Republic,2,False
2,Canada,1,True
3,USA,4,True
4,Qatar,1,False


In [21]:

host_goals_all = team_goals[team_goals["is_host"] == True]["goals"]
other_goals_all = team_goals[team_goals["is_host"] == False]["goals"]


In [22]:

# host teams only have 15 records in total, so all of them are used
host_sample = host_goals_all
other_sample = other_goals_all.sample(n=40, random_state=7)


In [23]:

print_stats(host_sample, "Host teams")
print_stats(other_sample, "Non-host teams")


Host teams
count: 15
mean: 2.0
median: 2.0
std dev: 1.46

Non-host teams
count: 40
mean: 1.58
median: 1.0
std dev: 1.5



In [24]:

ci_low, ci_high = get_ci(host_sample)
print("95% confidence interval for host teams mean goals:", round(ci_low, 2), "to", round(ci_high, 2))


95% confidence interval for host teams mean goals: 1.19 to 2.81


In [25]:

t_value, p_value = stats.ttest_ind(host_sample, other_sample, equal_var=False)
print("t value:", round(t_value, 3))
print("p value:", round(p_value, 4))


t value: 0.952
p value: 0.3497


In [26]:

if p_value < 0.05:
    print("There is a significant difference between host and non-host teams.")
else:
    print("There is no significant difference between host and non-host teams.")


There is no significant difference between host and non-host teams.


---
## Task 2: UEFA vs CONMEBOL (yellow cards per 90 minutes)

Population = all UEFA and CONMEBOL squads at the tournament. There are only 16 UEFA teams and 6 CONMEBOL teams, so all of them are used (a census, not a sample).

In [27]:

uefa_teams = ["Austria", "Belgium", "Bosnia\u2013Herz", "Croatia", "Czechia", "England",
              "France", "Germany", "Netherlands", "Norway", "Portugal", "Scotland",
              "Spain", "Switzerland", "Sweden", "T\u00fcrkiye"]


In [28]:

conmebol_teams = ["Argentina", "Brazil", "Colombia", "Ecuador", "Paraguay", "Uruguay"]
cards["cards_per_90"] = cards["CrdY"] / cards["90s"]


In [29]:

uefa_cards = cards[cards["Squad"].isin(uefa_teams)]["cards_per_90"]
conmebol_cards = cards[cards["Squad"].isin(conmebol_teams)]["cards_per_90"]


In [30]:

print_stats(uefa_cards, "UEFA")
print_stats(conmebol_cards, "CONMEBOL")


UEFA
count: 16
mean: 0.99
median: 0.96
std dev: 0.41

CONMEBOL
count: 6
mean: 1.69
median: 1.67
std dev: 0.17



In [31]:

ci_low, ci_high = get_ci(uefa_cards)
print("95% confidence interval for UEFA mean cards per 90:", round(ci_low, 2), "to", round(ci_high, 2))


95% confidence interval for UEFA mean cards per 90: 0.77 to 1.2


In [32]:

t_value, p_value = stats.ttest_ind(uefa_cards, conmebol_cards, equal_var=False)
print("t value:", round(t_value, 3))
print("p value:", round(p_value, 6))


t value: -5.763
p value: 1.3e-05


In [33]:

if p_value < 0.05:
    print("There is a significant difference between UEFA and CONMEBOL.")
else:
    print("There is no significant difference between UEFA and CONMEBOL.")


There is a significant difference between UEFA and CONMEBOL.


---
## Task 3: Goalkeeper save % - knockout teams vs group-stage-only teams

Population = all 48 team goalkeepers. 32 teams reached the knockout stage and 16 were knocked out in the group stage, so the full population is used for both groups.

In [34]:

knockout_rounds = ["Round of 32", "Round of 16", "Quarter-finals",
                    "Semi-finals", "Third-place match", "Final"]


In [35]:

knockout_matches = matches[matches["Round"].isin(knockout_rounds)]
knockout_teams = set(knockout_matches["home_team"]) | set(knockout_matches["away_team"])


In [36]:

gk_stats["reached_knockout"] = gk_stats["Squad"].isin(knockout_teams)


In [37]:

knockout_save = gk_stats[gk_stats["reached_knockout"] == True]["Save%"]
group_only_save = gk_stats[gk_stats["reached_knockout"] == False]["Save%"]


In [38]:

print_stats(knockout_save, "Knockout stage teams")
print_stats(group_only_save, "Group stage only teams")


Knockout stage teams
count: 32
mean: 67.99
median: 69.8
std dev: 12.57

Group stage only teams
count: 16
mean: 58.53
median: 58.95
std dev: 13.91



In [39]:

ci_low, ci_high = get_ci(knockout_save)
print("95% confidence interval for knockout teams mean save %:", round(ci_low, 2), "to", round(ci_high, 2))


95% confidence interval for knockout teams mean save %: 63.46 to 72.52


In [40]:

t_value, p_value = stats.ttest_ind(knockout_save, group_only_save, equal_var=False)
print("t value:", round(t_value, 3))
print("p value:", round(p_value, 4))


t value: 2.293
p value: 0.0297


In [41]:

if p_value < 0.05:
    print("There is a significant difference between the two groups.")
else:
    print("There is no significant difference between the two groups.")


There is a significant difference between the two groups.


---
## Task 4: Starters vs substitutes (goals + assists per 90 minutes)

Population = every player who played at least 45 minutes. Sample = 60 starters and 60 substitutes chosen at random.

In [42]:

goals_assists = player_stats[["Player", "Squad", "Gls", "Ast", "90s"]]
minutes_info = player_time[["Player", "Squad", "Starts", "Subs", "Min"]]


In [43]:

players = goals_assists.merge(minutes_info, on=["Player", "Squad"])
players = players[players["Min"] >= 45]


In [44]:

players["ga_per_90"] = (players["Gls"] + players["Ast"]) / players["90s"]


In [45]:

# label a player as starter if they started more games than they came on as sub
players = players[players["Starts"] != players["Subs"]]
players["role"] = np.where(players["Starts"] > players["Subs"], "Starter", "Substitute")


In [46]:

players["role"].value_counts()


role
Starter       615
Substitute    146
Name: count, dtype: int64

In [47]:

starter_all = players[players["role"] == "Starter"]["ga_per_90"]
sub_all = players[players["role"] == "Substitute"]["ga_per_90"]


In [48]:

starter_sample = starter_all.sample(n=60, random_state=7)
sub_sample = sub_all.sample(n=60, random_state=7)


In [49]:

print_stats(starter_sample, "Starters")
print_stats(sub_sample, "Substitutes")


Starters
count: 60
mean: 0.23
median: 0.0
std dev: 0.36

Substitutes
count: 60
mean: 0.35
median: 0.0
std dev: 0.64



In [50]:

ci_low, ci_high = get_ci(starter_sample)
print("95% confidence interval for starters mean goals+assists per 90:", round(ci_low, 2), "to", round(ci_high, 2))


95% confidence interval for starters mean goals+assists per 90: 0.14 to 0.33


In [51]:

t_value, p_value = stats.ttest_ind(starter_sample, sub_sample, equal_var=False)
print("t value:", round(t_value, 3))
print("p value:", round(p_value, 4))


t value: -1.242
p value: 0.2173


In [52]:

if p_value < 0.05:
    print("There is a significant difference between starters and substitutes.")
else:
    print("There is no significant difference between starters and substitutes.")


There is no significant difference between starters and substitutes.


Note: substitutes came out higher here, not lower. This can happen because a substitute who scores in a short amount of playing time gets a very high per-90 number. Only players with 45+ minutes were kept to reduce this, but it can still happen with a small sample.

---
## Summary of all 4 tasks

In [53]:

summary = pd.DataFrame({
    "Task": ["Host vs non-host goals", "UEFA vs CONMEBOL cards",
             "Knockout vs group-only save%", "Starters vs subs G+A per 90"],
    "Group A mean": [host_sample.mean(), uefa_cards.mean(), knockout_save.mean(), starter_sample.mean()],
    "Group B mean": [other_sample.mean(), conmebol_cards.mean(), group_only_save.mean(), sub_sample.mean()],
})


In [54]:

summary


,Task,Group A mean,Group B mean
0,Host vs non-host goals,2.000000,1.575000
1,UEFA vs CONMEBOL cards,0.985710,1.690147
2,Knockout vs group-only save%,67.993750,58.531250
3,Starters vs subs G+A per 90,0.233918,0.352063
